In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Set seed for reproducibility
np.random.seed(42)

# Lists for random selection
names = ["Joao Silva", "Maria", "Ana Costa", "Pedro", "Carlos Souza", "Julia", "Lucas", "Fernanda", "Robert", "Emma"]
products = ["Laptop", "Phone", "Mouse", "Keyboard", "Monitor", "Charger", "Tablet", "Headphones", "Camera", "Printer"]
regions = ["Sao Paulo", "SP", "Rio", "RJ", "NY", "New York", "TX", "Texas", "California", "CA"]
date_formats = ["%Y-%m-%d", "%m/%d/%Y", "%Y/%m/%d"]

# Generate 100 rows
data = []
for i in range(100):
    order_id = 1001 + i
    name = random.choice(names)
    if random.random() < 0.2:  # 20% chance of lowercase or typo
        name = name.lower() if random.random() < 0.5 else name.replace("a", "e", 1)
    product = random.choice(products) if random.random() > 0.05 else ""  # 5% missing
    if random.random() < 0.1:  # 10% lowercase
        product = product.lower()
    price = random.randint(20, 1500)
    if random.random() < 0.05:  # 5% text error
        price = random.choice(["thirty", "hundred", "ten"])
    elif random.random() < 0.02:  # 2% outlier
        price = random.choice([9999, -50])
    qty = random.randint(1, 10)
    if random.random() < 0.05:  # 5% negative
        qty = -qty
    elif random.random() < 0.05:  # 5% extreme
        qty = random.randint(100, 1000)
    elif random.random() < 0.1:  # 10% missing
        qty = ""
    base_date = datetime(2023, 1, 1) + timedelta(days=random.randint(0, 364))
    date_format = random.choice(date_formats)
    if random.random() < 0.02:  # 2% invalid date
        sale_date = "2023-13-01"
    else:
        sale_date = base_date.strftime(date_format)
    region = random.choice(regions)
    if random.random() < 0.1:  # 10% typo
        region = region.replace("a", "e", 1)

    data.append([order_id, name, product, price, qty, sale_date, region])

# Create DataFrame and save
df = pd.DataFrame(data, columns=["OrderID", "CustomerName", "Product", "Price", "Quantity", "SaleDate", "Region"])
df.to_csv("messy_sales_100.csv", index=False)
print("Dataset saved as 'messy_sales_100.csv'!")

Dataset saved as 'messy_sales_100.csv'!


In [5]:
df.head()

,OrderID,CustomerName,Product,Price,Quantity,SaleDate,Region
0,1001,Joao Silva,Phone,932,4,08/17/2023,TX
1,1002,Robert,camera,300,10,2023-03-30,NY
2,1003,Carlos Souza,Printer,thirty,7,09/04/2023,NY
3,1004,Maria,Laptop,1037,6,2023-10-10,SP
4,1005,Emma,Keyboard,1474,8,2023-03-25,Texas


In [21]:
blanks = df == ""  # Boolean mask
print(blanks.sum())  # How many per column

OrderID          0
CustomerName     0
Product          5
Price            0
Quantity        15
SaleDate         0
Region           0
dtype: int64


In [35]:
df.replace("", np.nan, inplace=True)

In [45]:
df_clean = df.dropna(subset='Product').copy()
df_clean

,OrderID,CustomerName,Product,Price,Quantity,SaleDate,Region
0,1001,Joao Silva,Phone,932,4.0,08/17/2023,TX
1,1002,Robert,camera,300,10.0,2023-03-30,NY
2,1003,Carlos Souza,Printer,thirty,7.0,09/04/2023,NY
3,1004,Maria,Laptop,1037,6.0,2023-10-10,SP
4,1005,Emma,Keyboard,1474,8.0,2023-03-25,Texas
...,...,...,...,...,...,...,...
95,1096,Carlos Souza,Monitor,412,5.0,2023/05/22,SP
96,1097,Emma,Camera,231,4.0,2023/04/06,New York
97,1098,Robert,Printer,195,4.0,03/17/2023,RJ
98,1099,Robert,Printer,1401,9.0,2023/10/12,New York


In [51]:
df_clean.Quantity = pd.to_numeric(df_clean.Quantity.fillna(0))
df_clean.Quantity.value_counts()

Quantity
 0.0      14
 10.0     12
 2.0      10
 6.0       9
 9.0       9
 4.0       8
 5.0       8
 7.0       5
 1.0       5
 3.0       5
 8.0       4
-6.0       2
-1.0       1
 263.0     1
 410.0     1
 711.0     1
Name: count, dtype: int64

In [65]:
df_clean.duplicated().value_counts()
df_clean.drop_duplicates()

,OrderID,CustomerName,Product,Price,Quantity,SaleDate,Region
0,1001,Joao Silva,Phone,932,4.0,08/17/2023,TX
1,1002,Robert,camera,300,10.0,2023-03-30,NY
2,1003,Carlos Souza,Printer,thirty,7.0,09/04/2023,NY
3,1004,Maria,Laptop,1037,6.0,2023-10-10,SP
4,1005,Emma,Keyboard,1474,8.0,2023-03-25,Texas
...,...,...,...,...,...,...,...
95,1096,Carlos Souza,Monitor,412,5.0,2023/05/22,SP
96,1097,Emma,Camera,231,4.0,2023/04/06,New York
97,1098,Robert,Printer,195,4.0,03/17/2023,RJ
98,1099,Robert,Printer,1401,9.0,2023/10/12,New York


In [73]:
df_clean.CustomerName = df_clean.CustomerName.str.title()
df_clean.CustomerName

0       Joao Silva
1           Robert
2     Carlos Souza
3            Maria
4             Emma
          ...     
95    Carlos Souza
96            Emma
97          Robert
98          Robert
99       Ana Costa
Name: CustomerName, Length: 95, dtype: object

In [77]:
customer_name_split = df_clean.CustomerName.str.split(' ')
customer_name_split

0       [Joao, Silva]
1            [Robert]
2     [Carlos, Souza]
3             [Maria]
4              [Emma]
           ...       
95    [Carlos, Souza]
96             [Emma]
97           [Robert]
98           [Robert]
99       [Ana, Costa]
Name: CustomerName, Length: 95, dtype: object

In [95]:
df_clean.CustomerName = customer_name_split.str.get(0)
df_clean['CustomerLastName'] = customer_name_split.str.get(1)
df_clean

,OrderID,CustomerName,Product,Price,Quantity,SaleDate,Region,CustomerLastName
0,1001,Joao,Phone,932,4.0,08/17/2023,TX,Silva
1,1002,Robert,camera,300,10.0,2023-03-30,NY,NaN
2,1003,Carlos,Printer,thirty,7.0,09/04/2023,NY,Souza
3,1004,Maria,Laptop,1037,6.0,2023-10-10,SP,NaN
4,1005,Emma,Keyboard,1474,8.0,2023-03-25,Texas,NaN
...,...,...,...,...,...,...,...,...
95,1096,Carlos,Monitor,412,5.0,2023/05/22,SP,Souza
96,1097,Emma,Camera,231,4.0,2023/04/06,New York,NaN
97,1098,Robert,Printer,195,4.0,03/17/2023,RJ,NaN
98,1099,Robert,Printer,1401,9.0,2023/10/12,New York,NaN


In [107]:
df_clean.Product = df_clean.Product.str.capitalize()
df_clean.Product.value_counts()

Product
Keyboard      14
Charger       12
Monitor       12
Phone         11
Laptop        11
Printer       10
Camera         7
Mouse          7
Tablet         7
Headphones     4
Name: count, dtype: int64

In [153]:
df_clean.Price[df_clean.Price.str.contains(r'\D', regex=True, na=False)]

2      thirty
5         ten
9         ten
12    hundred
22     thirty
29        ten
31    hundred
86        ten
Name: Price, dtype: object

In [175]:
price_fix = {'thirty': 30, 'ten': 10, 'hundred': 100}
df_clean['Price'] = df_clean['Price'].replace(price_fix)
df_clean['Price'] = pd.to_numeric(df_clean['Price'], errors='coerce')

In [187]:
df_clean = df_clean.reset_index(drop=True)
df_clean

,OrderID,CustomerName,Product,Price,Quantity,SaleDate,Region,CustomerLastName
0,1001,Joao,Phone,932,4.0,08/17/2023,TX,Silva
1,1002,Robert,Camera,300,10.0,2023-03-30,NY,NaN
2,1003,Carlos,Printer,30,7.0,09/04/2023,NY,Souza
3,1004,Maria,Laptop,1037,6.0,2023-10-10,SP,NaN
4,1005,Emma,Keyboard,1474,8.0,2023-03-25,Texas,NaN
...,...,...,...,...,...,...,...,...
90,1096,Carlos,Monitor,412,5.0,2023/05/22,SP,Souza
91,1097,Emma,Camera,231,4.0,2023/04/06,New York,NaN
92,1098,Robert,Printer,195,4.0,03/17/2023,RJ,NaN
93,1099,Robert,Printer,1401,9.0,2023/10/12,New York,NaN


In [189]:
df_clean['Region'].value_counts()

Region
SP            12
California    12
Sao Paulo     12
RJ            12
Rio           11
Texas          8
NY             7
New York       7
CA             7
TX             5
Celifornia     2
Name: count, dtype: int64

In [195]:
region_fix = {'SP': 'Sao Paulo', 'RJ': 'Rio', 'NY': 'New York', 'CA': 'California', 'TX': 'Texas', 'Celifornia': 'California'}
df_clean['Region'] = df_clean['Region'].replace(region_fix)
df_clean['Region'].value_counts()

Region
Sao Paulo     24
Rio           23
California    21
New York      14
Texas         13
Name: count, dtype: int64

In [201]:
df_clean

,OrderID,CustomerName,Product,Price,Quantity,SaleDate,Region,CustomerLastName
0,1001,Joao,Phone,932,4.0,08/17/2023,Texas,Silva
1,1002,Robert,Camera,300,10.0,2023-03-30,New York,NaN
2,1003,Carlos,Printer,30,7.0,09/04/2023,New York,Souza
3,1004,Maria,Laptop,1037,6.0,2023-10-10,Sao Paulo,NaN
4,1005,Emma,Keyboard,1474,8.0,2023-03-25,Texas,NaN
...,...,...,...,...,...,...,...,...
90,1096,Carlos,Monitor,412,5.0,2023/05/22,Sao Paulo,Souza
91,1097,Emma,Camera,231,4.0,2023/04/06,New York,NaN
92,1098,Robert,Printer,195,4.0,03/17/2023,Rio,NaN
93,1099,Robert,Printer,1401,9.0,2023/10/12,New York,NaN


In [205]:
df_clean['SaleDate'] = pd.to_datetime(df_clean['SaleDate'], errors='coerce')
df_clean

,OrderID,CustomerName,Product,Price,Quantity,SaleDate,Region,CustomerLastName
0,1001,Joao,Phone,932,4.0,2023-08-17,Texas,Silva
1,1002,Robert,Camera,300,10.0,NaT,New York,NaN
2,1003,Carlos,Printer,30,7.0,2023-09-04,New York,Souza
3,1004,Maria,Laptop,1037,6.0,NaT,Sao Paulo,NaN
4,1005,Emma,Keyboard,1474,8.0,NaT,Texas,NaN
...,...,...,...,...,...,...,...,...
90,1096,Carlos,Monitor,412,5.0,NaT,Sao Paulo,Souza
91,1097,Emma,Camera,231,4.0,NaT,New York,NaN
92,1098,Robert,Printer,195,4.0,2023-03-17,Rio,NaN
93,1099,Robert,Printer,1401,9.0,NaT,New York,NaN


In [211]:
df_clean = df_clean.sort_values('SaleDate').reset_index(drop=True)
df_clean

,OrderID,CustomerName,Product,Price,Quantity,SaleDate,Region,CustomerLastName
0,1026,Joao,Laptop,934,6.0,2023-01-02,Sao Paulo,Silva
1,1078,Carlos,Mouse,1127,10.0,2023-01-07,New York,Souza
2,1020,Maria,Keyboard,766,0.0,2023-01-08,New York,NaN
3,1024,Robert,Charger,659,1.0,2023-01-25,California,NaN
4,1098,Robert,Printer,195,4.0,2023-03-17,Rio,NaN
...,...,...,...,...,...,...,...,...
90,1094,Carlos,Mouse,750,0.0,NaT,Sao Paulo,Souza
91,1096,Carlos,Monitor,412,5.0,NaT,Sao Paulo,Souza
92,1097,Emma,Camera,231,4.0,NaT,New York,NaN
93,1099,Robert,Printer,1401,9.0,NaT,New York,NaN


In [219]:
df_clean['SaleDate'] = df_clean['SaleDate'].ffill()
df_clean

,OrderID,CustomerName,Product,Price,Quantity,SaleDate,Region,CustomerLastName
0,1026,Joao,Laptop,934,6.0,2023-01-02,Sao Paulo,Silva
1,1078,Carlos,Mouse,1127,10.0,2023-01-07,New York,Souza
2,1020,Maria,Keyboard,766,0.0,2023-01-08,New York,NaN
3,1024,Robert,Charger,659,1.0,2023-01-25,California,NaN
4,1098,Robert,Printer,195,4.0,2023-03-17,Rio,NaN
...,...,...,...,...,...,...,...,...
90,1094,Carlos,Mouse,750,0.0,2023-12-28,Sao Paulo,Souza
91,1096,Carlos,Monitor,412,5.0,2023-12-28,Sao Paulo,Souza
92,1097,Emma,Camera,231,4.0,2023-12-28,New York,NaN
93,1099,Robert,Printer,1401,9.0,2023-12-28,New York,NaN
